In [ ]:
#Parameters

Scenario = None
T_planet = None
M_planet = None
R_planet = None
G_planet = None
Z_planet = None
Age = None
Sep_p = None

folder_name = None
log_file = None
day = None

mol_index_start = None
mol_index_finish = None 

vmin= None
vmax = None
step = None

In [ ]:
# Cell 1: Import necessary libraries and set up paths
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import matplotlib

from scipy import interpolate
from scipy.stats import norm
from scipy.optimize import curve_fit
from scipy.linalg import svd, solve, det
from scipy.signal import savgol_filter

from exocrires import info
from exocrires import plotMatrix
from exocrires.spectra_processor import Spectra_fitting  
from exocrires.spectra_processor import processor_cross_correlation
from exocrires.spectra_processor import processor_likelihood_map

from astropy.io import fits

import glob
import tqdm
import time 
import os 
import shutil
import gc

# Define paths to data directories
font = {'size': 20}
plt.rcParams.update({'font.size': font['size']})

model_name='%s-%s-%s'%(T_planet,G_planet,Z_planet)

band='K2166'
distance=140 #pc

#sep_AU=82.6 #AU
#sep=sep_AU/distance #arcsec
#po=int(np.ceil(sep/0.059))

path_a = '/Users/richard/project_EXOCRIRES/' + folder_name + '/%s'%day 
path_b = '/Users/richard/project_EXOCRIRES'
path_file=path_a+'/planet/%s/%s'%(model_name, band)

simulation_log = pd.read_csv(path_b+'/%s'%log_file)


stack=np.load(path_file+'/ABBA_stack.npy') 
series=np.load(path_file+'/ABBA_series.npy')
stack_sub_badpix=np.load(path_file+'/stack_sub_badpix.npy')
fit_matrix=np.load(path_file+'/fit_matrix.npy')

default_vmin=-50
default_vmax=50
default_step=5
N_components=5
window=300

default_grids=np.arange(default_vmin, default_vmax, default_step)

default_model_matrix=np.load(path_file+'/model_matrix_combine_%s_%s_%s_%s_%s.npy'%(default_vmin, default_vmax, default_step, N_components, window))

# Ask the user to input vmin, vmax, and step

# Split the input into three parts and convert them to float or int
#vmin, vmax, step = map(float, user_input.split())

#Cut the model
full_grids = np.arange(default_vmin, default_vmax, default_step)

if [vmin, vmax, step] != [default_vmin, default_vmax, default_step]:

    
    mask = (full_grids >= vmin) & (full_grids < vmax)
    selected_indices = np.where(mask)[0]
    model_matrix_combine = default_model_matrix[selected_indices]
    grids = full_grids[selected_indices]

else:

    model_matrix_combine = default_model_matrix
    grids = default_grids



lambda_range=info.lambda_range
mol_name = ['CH4', 'CO', 'H2O']

# 4. Likelihood Calculation

## 4.1 Likelihood Calculation for the full spectrum (nominal model)

In [ ]:
target_pix = int(17 + Sep_p)


workflow_likelihood=processor_likelihood_map()
delta_log_likelihood_no_planet=np.zeros(shape=(stack_sub_badpix.shape[0], stack_sub_badpix.shape[1], len(grids)))

if os.path.exists(path_file+'/variance.npy') != True:
    
    output=workflow_likelihood.covariance_calculator(fit_matrix, series, series.shape[1])
    var=output[0]

    np.save(path_file+'/variance.npy', var)

else:
    var = np.load(path_file+'/variance.npy')

print ('The variance matrix is loaded.')



for order in range(2, stack_sub_badpix.shape[0]):


    # Clean the observation matrix and model matrices by replacing NaN or inf values with 0
    observation_matrix_clean = np.nan_to_num(stack_sub_badpix[order], nan=0.0, posinf=0.0, neginf=0.0)
    model_matrix_clean = np.nan_to_num(model_matrix_combine[:, [0,1], order, :, :], nan=0.0, posinf=0.0, neginf=0.0)
    model_matrix_noplanet_clean = np.nan_to_num(model_matrix_combine[:, 0, order, :, :], nan=0.0, posinf=0.0, neginf=0.0)
    
    obs = observation_matrix_clean[target_pix:target_pix+1, :]
    model = model_matrix_clean[:, :, target_pix:target_pix+1, :]
    model_no_planet = model_matrix_noplanet_clean[:, target_pix:target_pix+1, :]
    variance = var[order][target_pix:target_pix+1]

    # Perform likelihood map calculations
    Map = workflow_likelihood.likelihood_map(
        observation_matrix=obs,
        model_matrix=model,
        n_vgrids=len(grids),
        matrix_components=2,
        sigma_matrix=variance,
        prior=1,
        skip_velocity_grid_single_component=True
    )
    Map_noplanet = workflow_likelihood.likelihood_map(
        observation_matrix=obs,
        model_matrix=model_no_planet,
        n_vgrids=len(grids),
        matrix_components=1,
        sigma_matrix=variance,
        prior=1,
        skip_velocity_grid_single_component=True
    )
    delta_log_likelihood_no_planet[order][target_pix:target_pix+1] = Map - Map_noplanet
    #delta_log_likelihood_no_planet[order] = Map - Map_noplanet
    print('Order %s processed' % order)

#save the delta log likelihood map
#path_file=path_a+'/%s/planet/%s/%s'%(day,model_name,band)
#np.save(path_file+'/delta_log_likelihood_noplanet.npy', delta_log_likelihood_no_planet)

#clean the cache

gc.collect()


In [ ]:

mask=np.isfinite(Map)
Map[~mask]=np.nan

delta_log_lokelihood_tot_sum=np.nansum(delta_log_likelihood_no_planet, axis=0)

plotMatrix.plotMatrix(delta_log_lokelihood_tot_sum, grids, np.arange(0, delta_log_lokelihood_tot_sum.shape[0],1), 'System velocity (km/s)', 'spatial axis',\
     planet_posi=target_pix, scale='log', power=1)
plt.vlines(x=10, ymin=0, ymax=34, ls='--', colors='Gray', linewidth=2)


plt.tight_layout()      
plt.savefig(path_a+'/delta_likelihood_new_g.png', dpi=100)
plt.show()

# Save the delta log likelihood map
plt.savefig(path_file+'/likelihood_map_full_spectrum_new_g.png',dpi=150)

#Save the likelihood at the planet position with 10km/s to the simulation log 
likeli = delta_log_lokelihood_tot_sum[target_pix, np.abs(grids-10).argmin()]
simulation_log.loc[simulation_log['Age']==Age, 'Likelihood_full'] = likeli

#plot the delta likelihood at the planet position
plt.plot(grids, delta_log_lokelihood_tot_sum[target_pix])
plt.vlines(x=10, ymin=np.min(delta_log_lokelihood_tot_sum)-5, ymax=np.max(delta_log_lokelihood_tot_sum)+10, ls='--', colors='Gray', label=r'$v_{sys}=10km/s$')

plt.xlabel('System velocity (km/s)')
plt.ylabel(r'$\Delta lnL$')

plt.title('Planet Position')

plt.legend(loc='upper left', fontsize=10)
plt.tight_layout()


plt.savefig(path_a+'/delta_likelihood_planet_posi_new_g.png', dpi=100)

plt.show()

delta_likelihood_snr=delta_log_lokelihood_tot_sum/np.std(delta_log_lokelihood_tot_sum[:,0:5])

plotMatrix.plotMatrix(delta_likelihood_snr, grids, np.arange(0, delta_log_lokelihood_tot_sum.shape[0],1), 'System velocity (km/s)', 'spatial axis', planet_posi=(target_pix),scale='log', power=1)
plt.vlines(x=10, ymin=0, ymax=34, ls='--', colors='Gray', linewidth=2)
#plt.plot(Map[1][:,1000]-Map_noplanet[1][:,1000])
#plt.xlabel('spatial axis')
#plt.ylabel(r'$\Delta L$')
#plt.vlines(x=27, ymin=0, ymax=600, ls='--')

plt.tight_layout()
plt.savefig(path_a+'/delta_likelihood_snr_new_g.png', dpi=100)
plt.show()


plt.plot(grids, delta_likelihood_snr[target_pix])
plt.vlines(x=10, ymin=np.min(delta_likelihood_snr)-5, ymax=np.max(delta_likelihood_snr)+10, ls='--', colors='Gray', label=r'$v_{sys}=10km/s$')
plt.xlabel('System velocity (km/s)')
plt.ylabel(r'$SNR$')
plt.title('Planet Position')
plt.legend(loc='upper left', fontsize=10)
plt.tight_layout()

plt.show()

#save simulation log
np.save(path_a+'/likelihood_map_full_new_g.npy', delta_log_lokelihood_tot_sum)
simulation_log.to_csv(path_b+'/%s'%log_file, index=False)
print('Likelihood calculation for the nominal model completed. Simulation log saved to %s/%s'%(path_b, log_file))


## 4.2 Likelihood Calculation for the molecules 

In [ ]:
var=np.load(path_file+'/variance.npy')

for mol_index in range(mol_index_start, mol_index_finish):

    delta_log_likelihood_no_mol=np.zeros(shape=(stack_sub_badpix.shape[0], stack_sub_badpix.shape[1], len(grids)))

    workflow_likelihood=processor_likelihood_map()
    #output=workflow_likelihood.covariance_calculator(fit_matrix, series, series.shape[1])
    #var=output[0]

    print ('Processing molecule %s'%mol_name[mol_index-2])


    for order in range(2, stack_sub_badpix.shape[0]):



        # Clean the observation matrix and model matrices by replacing NaN or inf values with 0
        observation_matrix_clean = np.nan_to_num(stack_sub_badpix[order], nan=0.0, posinf=0.0, neginf=0.0)
        model_matrix_clean = np.nan_to_num(model_matrix_combine[:, [0,1], order, :, :], nan=0.0, posinf=0.0, neginf=0.0)
        model_matrix_nomol_clean = np.nan_to_num(model_matrix_combine[:, [0, mol_index], order, :, :], nan=0.0, posinf=0.0, neginf=0.0)

        
        obs = observation_matrix_clean[target_pix:target_pix+1, :]
        model = model_matrix_clean[:, :, target_pix:target_pix+1, :]
        model_nomol = model_matrix_nomol_clean[:, :, target_pix:target_pix+1, :]
        variance = var[order][target_pix:target_pix+1]

        # Perform likelihood map calculations
        Map = workflow_likelihood.likelihood_map(
            observation_matrix=obs,
            model_matrix=model,
            n_vgrids=len(grids),
            matrix_components=2,
            sigma_matrix=variance,
            prior=1,
            skip_velocity_grid_single_component=True
        )
        Map_noplanet = workflow_likelihood.likelihood_map(
            observation_matrix=obs,
            model_matrix=model_nomol,
            n_vgrids=len(grids),
            matrix_components=2,
            sigma_matrix=variance,
            prior=1,
            skip_velocity_grid_single_component=True
        )

        delta_log_likelihood_no_mol[order][target_pix:target_pix+1] = Map - Map_noplanet
        #delta_log_likelihood_no_mol[order] = Map - Map_noplanet

    #save the delta log likelihood map
    #path_file=path_a+'/%s/planet/%s/%s'%(day,model_name,band)
    #np.save(path_file+'/delta_log_likelihood_no_ch4.npy', delta_log_likelihood_no_mol)

    delta_log_lokelihood_sum=np.nansum(delta_log_likelihood_no_mol, axis=0)

    plotMatrix.plotMatrix(delta_log_lokelihood_sum, grids, np.arange(0, delta_log_lokelihood_sum.shape[0],1), 'System velocity (km/s)', 'spatial axis', planet_posi=target_pix, scale='log', power=1)
    plt.vlines(x=10, ymin=0, ymax=34, ls='--', colors='Gray', linewidth=2)
    #plt.plot(Map[1][:,1000]-Map_noplanet[1][:,1000])
    #plt.xlabel('spatial axis')
    #plt.ylabel(r'$\Delta L$')
    #plt.vlines(x=27, ymin=0, ymax=600, ls='--')

    plt.tight_layout()
    plt.savefig(path_a+'/delta_likelihood_no_%s_new_g.png'%mol_name[mol_index-2], dpi=100)
    plt.show()


    plt.plot(grids, delta_log_lokelihood_sum[target_pix])
    plt.vlines(x=10, ymin=np.min(delta_log_lokelihood_sum)-5, ymax=np.max(delta_log_lokelihood_sum)+10, ls='--', colors='Gray', label=r'$v_{sys}=10km/s$')

    plt.xlabel('System veloctiy (km/s)')
    plt.ylabel(r'$\Delta lnL$')

    plt.title('Planet Position')

    plt.legend(loc='upper left', fontsize=10)
    plt.tight_layout()


    plt.savefig(path_a+'/delta_likelihood_planet_posi_%s_new_g.png'%mol_name[mol_index-2], dpi=100)

    plt.show()

    delta_likelihood_snr=delta_log_lokelihood_sum/np.std(delta_log_lokelihood_sum[:,0:5])

    plotMatrix.plotMatrix(delta_likelihood_snr, grids, np.arange(0, delta_log_lokelihood_tot_sum.shape[0],1), 'System velocity (km/s)', 'spatial axis', planet_posi=target_pix, scale='log', power=1)
    plt.vlines(x=10, ymin=0, ymax=34, ls='--', colors='Gray', linewidth=2)
    #plt.plot(Map[1][:,1000]-Map_noplanet[1][:,1000])
    #plt.xlabel('spatial axis')
    #plt.ylabel(r'$\Delta L$')
    #plt.vlines(x=27, ymin=0, ymax=600, ls='--')

    plt.tight_layout()
    plt.savefig(path_a+'/delta_likelihood_snr_%s_new_g.png'%mol_name[mol_index-2], dpi=100)
    plt.show()


    plt.plot(grids, delta_likelihood_snr[target_pix])
    plt.vlines(x=10, ymin=np.min(delta_likelihood_snr)-5, ymax=np.max(delta_likelihood_snr)+10, ls='--', colors='Gray', label=r'$v_{sys}=10km/s$')
    plt.xlabel('System veloctiy (km/s)')
    plt.ylabel(r'$SNR$')
    plt.title('Planet Position')
    plt.legend(loc='upper left', fontsize=10)
    plt.tight_layout()

    plt.show()


    likeli = delta_log_lokelihood_sum[target_pix, np.abs(grids-10).argmin()]

    simulation_log.loc[simulation_log['Age']==Age, 'Likelihood_%s'%mol_name[mol_index-2]] = likeli

    np.save(path_a+'/likelihood_map_%s_new_g.npy'%mol_name[mol_index-2], delta_log_lokelihood_sum)



    #clean the cache
    gc.collect()
    plt.close('all')

#save the simulation log
simulation_log.to_csv(path_b+'/%s'%log_file, index=False)
print('Likelihood calculation for molecules completed. Simulation log saved to %s/%s.csv'%(path_file, log_file))
